In [ ]:
from pathlib import Path as _OutputPath

def _prefix_output(path):
    if path is None:
        return None
    p = _OutputPath(path)
    stem = p.stem.removesuffix("_autor")
    name = stem + p.suffix
    if name.startswith("autor_"):
        return path
    result = p.with_name("autor_" + name)
    return result if isinstance(path, _OutputPath) else str(result)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import sem
from scipy.signal import welch
import yaml
from fooof import FOOOF
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from datetime import datetime
import imageio
import numpy as np
import pandas as pd
from cerebellum.analysis.structure_analysis import StructureReport_v2
import cerebellum
from bsb import from_storage

# routines

In [ ]:
def safe_load_yaml(filepath):
    with open(filepath, 'r') as f:
        data = yaml.safe_load(f)
    if data is None:
        raise ValueError(f"❌ Error: file {filepath} is empty or invalid.")
    return data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os

def plot_metrics_merge_from_df_summary(df_summary, xvar, label_suffix, hue="dfType", outname="merged_structures"):
    if df_summary.empty or xvar not in df_summary.columns:
        print(f"⚠️ Variabile '{xvar}' non trovata in df_summary")
        return

    # === Crea la cartella se non esiste ===
    plot_dir = "./tempPlots"
    os.makedirs(plot_dir, exist_ok=True)

    # --- Plot 1: metriche strutturali globali ---
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metrics_global = [
        ("total_cells", "Total cells"),
        ("mean_density", "Mean density [μm⁻³]"),
        ("total_synapses", "Total synapses"),
        ("mean_syn_per_pair", "Synapses per pair"),
        ("mean_convergence", "Mean convergence"),
        ("mean_divergence", "Mean divergence")
    ]
    df_global = df_summary.drop_duplicates(subset=["dfType", xvar])

    for ax, (col, title) in zip(axes.flat, metrics_global):
        if col in df_summary.columns:
            sns.lineplot(data=df_global, x=xvar, y=col, hue=hue, ax=ax, marker="o", palette=['blue', 'red'])
            ax.set_title(title)
            ax.set_xlabel(label_suffix)
            ax.set_ylabel(col.replace("_", " ").capitalize())
            ax.grid(True)

    plt.tight_layout()
    fname_global = f"{outname}_summary_by_{xvar}_with_{hue}_autor.png"
    plt.savefig(_prefix_output(os.path.join(plot_dir, fname_global)), dpi=300)
    plt.close()
    print(f"📈 Salvato: {fname_global}")

    # --- Plot 2: metriche per tipo di cellula ---
    metrics_cells = ["count", "density"]
    if "cell_type" in df_summary.columns:
        df_cells = df_summary.dropna(subset=["cell_type"])
        for metric in metrics_cells:
            if metric in df_cells.columns:
                g = sns.FacetGrid(df_cells, col="cell_type", col_wrap=6, hue=hue, palette=['b', 'r'],
                                  sharey=False, sharex=False, height=4)
                g.map_dataframe(sns.lineplot, x=xvar, y=metric, marker="o")
                g.set_titles("{col_name}")
                g.set_axis_labels(label_suffix, metric.capitalize())
                for ax in g.axes.flat:
                    ax.grid(True)
                g.add_legend()
                g.fig.suptitle(f"Cell types – {metric} by {label_suffix}", fontsize=16)
                g.fig.tight_layout()
                g.fig.subplots_adjust(top=0.9)
                fname_cell = f"cells_{metric}_by_type_by_{xvar}_with_{hue}_autor.png"
                g.savefig(_prefix_output(os.path.join(plot_dir, fname_cell)), dpi=300)
                #plt.close()
                print(f"📈 Salvato: {fname_cell}")

    # --- Plot 3: metriche per tipo di connessione ---
    metrics_conns = ["synapses", "syn_per_pair", "convergence", "divergence", "weight", 
                     "effective_syn_per_pair", "effective_synapses",
                     "effective_convergence", "effective_divergence"]

    if "connection" in df_summary.columns:
        df_conns = df_summary.dropna(subset=["connection"])
        for metric in metrics_conns:
            if metric in df_conns.columns:
                g = sns.FacetGrid(df_conns, col="connection", col_wrap=8, hue=hue, palette=['b', 'r'], 
                                  sharey=False, sharex=False, height=4)
                g.map_dataframe(sns.lineplot, x=xvar, y=metric, marker="o")
                g.set_titles("{col_name}")
                g.set_axis_labels(label_suffix, metric.replace("_", " ").capitalize())
                for ax in g.axes.flat:
                    ax.grid(True)
                g.add_legend()
                g.fig.suptitle(f"Connectivity – {metric} by {label_suffix}", fontsize=16)
                g.fig.tight_layout()
                g.fig.subplots_adjust(top=0.9)
                fname_conn = f"connections_{metric}_by_type_by_{xvar}_with_{hue}_autor.png"
                g.savefig(_prefix_output(os.path.join(plot_dir, fname_conn)), dpi=300)
                #plt.close()
                print(f"📈 Salvato: {fname_conn}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os

def plot_metrics_merge(merged_df, xvar, label_suffix, hue="dfType", outname="merged_structures"):
    if merged_df.empty or xvar not in merged_df.columns:
        print(f"⚠️ Variabile '{xvar}' non trovata in merged_df")
        return

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metrics = [
        ("total_cells", "Total cells"),
        ("mean_density", "Mean density [μm⁻³]"),
        ("total_synapses", "Total synapses"),
        ("mean_syn_per_pair", "Synapses per pair"),
        ("mean_convergence", "Mean convergence"),
        ("mean_divergence", "Mean divergence")
    ]
    for ax, (col, title) in zip(axes.flat, metrics):
        if col in merged_df.columns:
            sns.lineplot(data=merged_df, x=xvar, y=col, hue=hue, ax=ax, marker="o")
            ax.set_title(title)
            ax.set_xlabel(label_suffix)
            ax.set_ylabel(col.replace("_", " ").capitalize())
            ax.grid(True)
    plt.tight_layout()
    #filename = f"{outname}_all_panels_by_{xvar}_with_hue_{hue}.png"
    #plt.savefig(os.path.join(plot_dir, filename), dpi=300)
    #plt.close()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os

def plot_metrics_merge(merged_df, df_cells, df_conns, xvar, label_suffix, hue="dfType", outname="merged_structures"):
    if merged_df.empty or xvar not in merged_df.columns:
        print(f"⚠️ Variabile '{xvar}' non trovata in merged_df")
        return

    # --- Plot 1: Metriche strutturali globali ---
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metrics_global = [
        ("total_cells", "Total cells"),
        ("mean_density", "Mean density [μm⁻³]"),
        ("total_synapses", "Total synapses"),
        ("mean_syn_per_pair", "Synapses per pair"),
        ("mean_convergence", "Mean convergence"),
        ("mean_divergence", "Mean divergence")
    ]
    for ax, (col, title) in zip(axes.flat, metrics_global):
        if col in merged_df.columns:
            sns.lineplot(data=merged_df, x=xvar, y=col, hue=hue, ax=ax, marker="o")
            ax.set_title(title)
            ax.set_xlabel(label_suffix)
            ax.set_ylabel(col.replace("_", " ").capitalize())
            ax.grid(True)
    plt.tight_layout()
    fname_global = f"{outname}_summary_by_{xvar}_with_{hue}_autor.png"
    plt.savefig(_prefix_output(os.path.join(plot_dir, fname_global)), dpi=300)
    plt.close()
    print(f"📈 Salvato: {fname_global}")

    # --- Plot 2: Metriche per tipo di cellula ---
    metrics_cells = ["count", "density"]
    for metric in metrics_cells:
        if metric in df_cells.columns:
            g = sns.FacetGrid(df_cells, col="cell_type", col_wrap=4, hue=hue,
                              sharey=False, sharex=False, height=4)
            g.map_dataframe(sns.lineplot, x=xvar, y=metric, marker="o")
            g.set_titles("{col_name}")
            g.set_axis_labels(label_suffix, metric.capitalize())
            for ax in g.axes.flat:
                ax.grid(True)
            g.add_legend()
            g.fig.suptitle(f"Cell types – {metric} by {label_suffix}", fontsize=16)
            g.fig.tight_layout()
            g.fig.subplots_adjust(top=0.9)
            fname_cell = f"cells_{metric}_by_type_by_{xvar}_with_{hue}_autor.png"
            g.savefig(_prefix_output(os.path.join(plot_dir, fname_cell)))
            plt.close()
            print(f"📈 Salvato: {fname_cell}")

    # --- Plot 3: Metriche per tipo di connessione ---
    metrics_conns = ["synapses", "syn_per_pair", "convergence", "divergence"]
    for metric in metrics_conns:
        if metric in df_conns.columns:
            g = sns.FacetGrid(df_conns, col="connection", col_wrap=4, hue=hue,
                              sharey=False, sharex=False, height=4)
            g.map_dataframe(sns.lineplot, x=xvar, y=metric, marker="o")
            g.set_titles("{col_name}")
            g.set_axis_labels(label_suffix, metric.replace("_", " ").capitalize())
            for ax in g.axes.flat:
                ax.grid(True)
            g.add_legend()
            g.fig.suptitle(f"Connectivity – {metric} by {label_suffix}", fontsize=16)
            g.fig.tight_layout()
            g.fig.subplots_adjust(top=0.9)
            fname_conn = f"connections_{metric}_by_type_by_{xvar}_with_{hue}_autor.png"
            g.savefig(_prefix_output(os.path.join(plot_dir, fname_conn)))
            plt.close()
            print(f"📈 Salvato: {fname_conn}")


In [ ]:
def compute_delta_fr_with_percent_and_zscore(
    df,
    group_col='pop',
    value_col='firing_rate_mean',
    std_col='firing_rate_std',
    sem_col='firing_rate_sem',
    k_col='k',
    delta_col='delta_fr_mean',
    delta_pct_col='delta_fr_percent',
    delta_std_col='delta_fr_std',
    delta_sem_col='delta_fr_sem',
    zscore_col='zscore_fr'
):
    """
    Aggiunge colonne:
    - delta assoluto e percentuale (rispetto a k=1)
    - variazione di std e sem
    - z-score rispetto al valore a k=1

    Parametri:
    - df: DataFrame con colonne firing_rate_* e pop, k
    - group_col: tipo gruppo (es. 'pop')
    - value_col: media (es. 'firing_rate_mean')
    - std_col, sem_col: deviazione standard, sem
    - k_col: colonna del grado di atrofia (k=1 = nessuna atrofia)
    - *_col: nomi delle nuove colonne da aggiungere

    Ritorna:
    - df esteso con colonne delta + zscore
    """
    df = df.copy()

    # Valori baseline per ogni gruppo (k = 1)
    baseline_mean = df[df[k_col] == 1].groupby(group_col)[value_col].mean().to_dict()
    baseline_std = df[df[k_col] == 1].groupby(group_col)[std_col].mean().to_dict()
    baseline_sem = df[df[k_col] == 1].groupby(group_col)[sem_col].mean().to_dict()

    # Delta assoluto
    df[delta_col] = df.apply(lambda row: row[value_col] - baseline_mean.get(row[group_col], np.nan), axis=1)

    # Delta percentuale
    def compute_pct(row):
        base = baseline_mean.get(row[group_col], np.nan)
        return 100 * (row[value_col] - base) / base if pd.notnull(base) and base != 0 else np.nan
    df[delta_pct_col] = df.apply(compute_pct, axis=1)

    # Delta std
    df[delta_std_col] = df.apply(lambda row: row[std_col] - baseline_std.get(row[group_col], np.nan), axis=1)

    # Delta sem
    df[delta_sem_col] = df.apply(lambda row: row[sem_col] - baseline_sem.get(row[group_col], np.nan), axis=1)

    # Z-score
    def compute_z(row):
        mean = baseline_mean.get(row[group_col], np.nan)
        std = baseline_std.get(row[group_col], np.nan)
        return (row[value_col] - mean) / std if pd.notnull(mean) and pd.notnull(std) and std != 0 else np.nan
    df[zscore_col] = df.apply(compute_z, axis=1)

    return df


In [ ]:
def plotFOOOFglobal(df_summary, hue=None):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np
    from scipy.stats import spearmanr, pearsonr
    from pathlib import Path

    ordered_cells = ["mossy", "granule", "golgi", "basket", "stellate", "purkinje"]
    cell_types = [c for c in ordered_cells if c in df_summary["pop"].unique()]

    def extract_fooof_features(df):
        return pd.DataFrame({
            "Atrophy Level (%)": df["Atrophy Level (%)"],
            "pop": df["pop"],
            "offset": df["FOOOF_aperiodic"].apply(lambda x: x[0] if isinstance(x, list) else np.nan),
            "slope": df["FOOOF_aperiodic"].apply(lambda x: x[1] if isinstance(x, list) else np.nan),
            "n_peaks": df["FOOOF_n_peaks"],
            hue: df[hue] if hue else None
        })

    df_fooof = extract_fooof_features(df_summary)

    features = ["offset", "slope"]
    n_features = len(features)
    fig, axes = plt.subplots(nrows=n_features, ncols=len(cell_types) + 1, figsize=(20, 10), sharey='row')
    axes = np.array(axes)

    def plot_feature(ax, df, feature, label):
        if hue:
            palette = sns.color_palette("husl", len(df[hue].dropna().unique()))
            sns.scatterplot(data=df, x="Atrophy Level (%)", y=feature, hue=hue, ax=ax, palette=palette, alpha=0.7)
            sns.lineplot(data=df, x="Atrophy Level (%)", y=feature, hue=hue, ax=ax, palette=palette,
                         estimator='mean', errorbar=None, legend=False, linewidth=1.5)

            grouped = df.groupby(hue)
            for i, (hval, group) in enumerate(grouped):
                x = group["Atrophy Level (%)"]
                y = group[feature]
                if len(x) >= 2:
                    rho, p_rho = spearmanr(x, y, nan_policy='omit')
                    ax.text(0.01, 0.95 - 0.05 * i, f"{hval}: ρ={rho:.2f}, p={p_rho:.1e}",
                            transform=ax.transAxes, fontsize=8, color=palette[i])
        else:
            x = df["Atrophy Level (%)"]
            y = df[feature]
            sns.scatterplot(x=x, y=y, ax=ax, color="black")
            sns.regplot(x=x, y=y, ax=ax, scatter=False, order=1, line_kws={"color": "black"}, ci=None)
            if 0 in x.values:
                y0 = y[x == 0].values
                if len(y0) > 0 and not np.isnan(y0[0]):
                    ax.axhline(y=y0[0], linestyle="--", linewidth=1.5, alpha=0.5, c='k')
            rho, p_rho = spearmanr(x, y)
            r, p_r = pearsonr(x, y)
            ax.set_title(f"{label}\nρ={rho:.2f}, p={p_rho:.1e} \n r={r:.2f}, p={p_r:.1e}")

        ax.set_xlabel("Atrophy Level (%)")
        if label == "All":
            ax.set_ylabel(feature.capitalize())
        else:
            ax.set_ylabel("")

    for i, feature in enumerate(features):
        plot_feature(axes[i, 0], df_fooof, feature, "All")
        for j, cell in enumerate(cell_types):
            df_cell = df_fooof[df_fooof["pop"] == cell]
            plot_feature(axes[i, j + 1], df_cell, feature, cell.capitalize())

    fig.suptitle("FOOOF Features vs Atrophy Level", fontsize=18, fontweight='bold', y=1.1)
    fig.tight_layout()
    plt.subplots_adjust(hspace=0.5, top=0.92)

    # output_path = Path("B_collective_results/2.functions/fooof_summary_plot_autor.png")
    # output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path = Path("./tempPlots/fooof_summary_plot_autor.png")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(_prefix_output(output_path), dpi=300)
    print(f"💾 Plot salvato in: {output_path}")

    plt.show()


In [ ]:
def plotFR(df_summary, hue=None, y_col='firing_rate_mean', yerr_col='firing_rate_sem', save=True):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd
    import os
    from scipy.stats import spearmanr

    cell_color_map = {
        "mossy_fibers": "purple",
        "granule": "gold",
        "golgi": "green",
        "basket": "blue",
        "stellate": "pink",
        "purkinje": "brown",
    }

    ordered_cells = [ "granule", "golgi", "basket", "stellate", "purkinje", "dcn_i", "dcn_p"] #"mossy_fibers", "glomerulus", 
    cell_types = [c for c in ordered_cells if c in df_summary["pop"].unique()]
    excitatory = ["granule"]
    inhibitory = ["golgi"]

    def compute_summary(df, subset, hue=None):
        filtered = df[df["pop"].isin(subset)]
        if hue:
            grp = filtered.groupby(["SZ Factor (%)", hue]).agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        else:
            grp = filtered.groupby("SZ Factor (%)").agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        return grp

    df_all = compute_summary(df_summary, cell_types, hue)
    df_exc = compute_summary(df_summary, excitatory, hue)
    df_inh = compute_summary(df_summary, inhibitory, hue)

    if not hue:
        df_ratio = pd.merge(df_exc, df_inh, on="SZ Factor (%)", suffixes=("_exc", "_inh"))
    else:
        df_ratio = pd.merge(df_exc, df_inh, on=["SZ Factor (%)", hue], suffixes=("_exc", "_inh"))

    df_ratio["ratio_EI"] = np.where(
        np.isfinite(df_ratio["y_mean_exc"] / df_ratio["y_mean_inh"]),
        df_ratio["y_mean_exc"] / df_ratio["y_mean_inh"],
        0)

    fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(17, 7), sharex=0, sharey=0)
    fig.subplots_adjust(hspace=0.6, wspace=0.6)  # Aumenta lo spazio verticale e orizzontale tra i pannelli
    axes = axes.flatten()

    def plot_with_corr(ax, df, x_col, y_col, yerr_col=None, label="", hue=None):
        if hue:
            hue_levels = df[hue].unique()
            #palette = sns.color_palette("husl", len(hue_levels))
            palette = {"High Resilience": "red", "Low Resilience": "blue"}
            for i, h in enumerate(hue_levels):
                d = df[df[hue] == h].sort_values(by=x_col)
                x = d[x_col].values
                y = d[y_col].values
                yerr = d[yerr_col].values if yerr_col and yerr_col in d.columns else None
                #color = palette[i]
                color = palette.get(h, f"C{i}")  # fallback color

                ax.plot(x, y, marker='o', label=f"{h}", color=color)

                if yerr is not None:
                    ax.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.2)
                rho, p_rho = spearmanr(x, y)
                #ax.text(0.01, 0.95 - 0.05 * i, f"{h}: ρ={rho:.2f}, p={p_rho:.1e}",
                        #transform=ax.transAxes, fontsize=9, color=color)
                if 0 in x:
                    baseline_y = y[x == 0][0]
                    ax.axhline(y=baseline_y, color="black", linestyle="--", linewidth=1.5, alpha=0.5)

            ax.set_title(f"{label}", fontsize=14, fontweight='bold')
        else:
            d = df.sort_values(by=x_col)
            x = d[x_col].values
            y = d[y_col].values
            yerr = d[yerr_col].values if yerr_col else None
            ax.plot(x, y, marker='o', color="black", label=label)

            if yerr is not None:
                ax.fill_between(x, y - yerr, y + yerr, color="black", alpha=0.2)
            rho, p_rho = spearmanr(x, y)
            #ax.set_title(f"{label}\nρ={rho:.2f}, p={p_rho:.1e}")
            if 0 in x:
                baseline_y = y[x == 0][0]
                ax.axhline(y=baseline_y, color="black", linestyle="--", linewidth=1.5, alpha=0.5)

        ax.set_xlabel("SZ Factor (%)")
        ylabel = r"$\Delta f$ (Hz)" #if y_col == "delta_fr_mean" else y_col.replace("_", " ").capitalize()
        ax.set_ylabel(ylabel)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plot_with_corr(axes[0], df_all, "SZ Factor (%)", "y_mean", "y_sem", "All cells", hue)
    used_slots = [0]

    for i, cell in enumerate(cell_types):
        cell_df = df_summary[df_summary["pop"] == cell]
        grp = cell_df.copy()
        if hue:
            grp = grp.groupby(["SZ Factor (%)", hue]).agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in grp.columns else (y_col, "sem")
            ).reset_index()
        else:
            grp = grp.groupby("SZ Factor (%)").agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in grp.columns else (y_col, "sem")
            ).reset_index()

        slot_index = 1 + i
        used_slots.append(slot_index)
        plot_with_corr(axes[slot_index], grp, "SZ Factor (%)", "y_mean", "y_sem", cell.capitalize(), hue)

    for j, ax in enumerate(axes):
        if j not in used_slots:
            ax.set_visible(False)

    if hue:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, title='Cell Resilience', loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.1), fontsize=12)


    if save:
        os.makedirs("./tempPlots", exist_ok=True)
        fname = f"./tempPlots/firing_rate_{y_col}_autor.png"
        plt.savefig(_prefix_output(fname), dpi=100)
        print(f"📈 Salvato: {fname}")

    plt.show()
    return df_exc, df_inh


In [ ]:
def plotFR_updated_v3(df_summary, hue=None, y_col='firing_rate_mean', yerr_col='firing_rate_sem', save=True):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os
    from string import ascii_uppercase as _AZ

    pretty_names = {
        "granule": "Granule Cells",
        "golgi": "Golgi Cells",
        "purkinje": "Purkinje Cells",
        "dcn_p": "DCN-P",
    }

    def compute_summary(df, subset, hue=None):
        filtered = df[df["pop"].isin(subset)]
        if hue:
            grp = filtered.groupby(["SZ Factor (%)", hue]).agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        else:
            grp = filtered.groupby("SZ Factor (%)").agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        return grp

    def plot_with_corr(ax, df, x_col, y_col, yerr_col=None, label="", hue=None):
        if hue:
            hue_levels = df[hue].unique()
            palette = {
                "With Resilience": "red",
                "Without Resilience": "blue",
                None: "black"
            }
            for i, h in enumerate(hue_levels):
                d = df[df[hue] == h].sort_values(by=x_col)
                x, y = d[x_col].values, d[y_col].values
                yerr = np.abs(d[yerr_col].values) if yerr_col and yerr_col in d.columns else None
                color = palette.get(h, f"C{i}")
                ax.errorbar(x, y, yerr=yerr, fmt='o-', label=f"{h}", color=color, capsize=4)
                if len(x) >= 3:
                    coeffs = np.polyfit(x, y, deg=5)
                    x_fit = np.linspace(min(x), max(x), 200)
                    ax.plot(x_fit, np.polyval(coeffs, x_fit), linestyle='-', linewidth=10, color=color, alpha=0.2)
                if 0 in x:
                    ax.axhline(y=y[x == 0][0], color="black", linestyle="--", linewidth=1.5, alpha=0.5)
        else:
            d = df.sort_values(by=x_col)
            x, y = d[x_col].values, d[y_col].values
            yerr = d[yerr_col].values if yerr_col else None
            ax.errorbar(x, y, yerr=yerr, fmt='o-', color="black", label=label, capsize=4)
            if len(x) >= 3:
                coeffs = np.polyfit(x, y, deg=5)
                x_fit = np.linspace(min(x), max(x), 200)
                ax.plot(x_fit, np.polyval(coeffs, x_fit), linestyle='--', color="black", alpha=1)
            if 0 in x:
                ax.axhline(y=y[x == 0][0], color="black", linestyle="--", linewidth=1.5, alpha=0.5)

        ax.set_title(label, fontsize=14, fontweight='bold')
        ax.set_xlabel("SZ Factor (%)")
        ax.set_ylabel(r"$\Delta f$ (Hz)")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # Escludi dcn_i
    df_summary = df_summary[df_summary["pop"] != "dcn_i"]

    df_all = compute_summary(df_summary, df_summary["pop"].unique(), hue)
    df_exc = compute_summary(df_summary, ["granule"], hue)
    df_inh = compute_summary(df_summary, ["golgi"], hue)

    # Layout 3x2
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(11, 13))
    fig.subplots_adjust(hspace=0.5, wspace=0.3)
    axes = axes.flatten()

    # Aggregazione ML interneuroni
    df_ml_inh = df_summary[df_summary["pop"].isin(["basket", "stellate"])]
    if hue:
        df_ml_inh_grp = df_ml_inh.groupby(["SZ Factor (%)", hue]).agg(
            y_mean=(y_col, "mean"),
            y_sem=(yerr_col, "mean") if yerr_col in df_ml_inh.columns else (y_col, "sem")
        ).reset_index()
    else:
        df_ml_inh_grp = df_ml_inh.groupby("SZ Factor (%)").agg(
            y_mean=(y_col, "mean"),
            y_sem=(yerr_col, "mean") if yerr_col in df_ml_inh.columns else (y_col, "sem")
        ).reset_index()

    # Lista dei plot
    plot_data = [
        (df_all, "All cells"),
        (compute_summary(df_summary, ["granule"], hue), pretty_names["granule"]),
        (compute_summary(df_summary, ["golgi"], hue), pretty_names["golgi"]),
        (df_ml_inh_grp, "MLIs"),
        (compute_summary(df_summary, ["purkinje"], hue), pretty_names["purkinje"]),
        (compute_summary(df_summary, ["dcn_p"], hue), pretty_names["dcn_p"]),
    ]

    # Aggiungi lettere minuscole in grassetto
    for i, (ax, (df_plot, label)) in enumerate(zip(axes, plot_data)):
        plot_with_corr(ax, df_plot, "SZ Factor (%)", "y_mean", "y_sem", label, hue)
        ax.text(-0.15, 1.02, f"{_AZ[i].lower()}",
                transform=ax.transAxes,
                fontsize=14, fontweight='bold', 
                va='top', ha='left')

    if hue:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.97), fontsize=12)

    if save:
        os.makedirs("./tempPlots", exist_ok=True)
        fname = f"./tempPlots/firing_rate_{y_col}_autor.png"
        plt.savefig(_prefix_output(fname), dpi=200, bbox_inches="tight")
        print(f"📈 Salvato: {fname}")

    # Figura correlazione PC–DCN-P
    purk = df_summary[df_summary["pop"] == "purkinje"]
    dcnp = df_summary[df_summary["pop"] == "dcn_p"]

    fig2, ax2 = plt.subplots(figsize=(6, 5))
    hue_levels = df_summary[hue].unique() if hue else [None]
    palette = {
        "With Resilience": "red",
        "Without Resilience": "blue",
        None: "black"
    }

    for h in hue_levels:
        purk_sub = purk[purk[hue] == h] if h else purk
        dcnp_sub = dcnp[dcnp[hue] == h] if h else dcnp
        merged = pd.merge(
            purk_sub[["SZ Factor (%)", y_col]],
            dcnp_sub[["SZ Factor (%)", y_col]],
            on="SZ Factor (%)",
            suffixes=("_purk", "_dcnp")
        )
        x, y = merged[f"{y_col}_purk"].values, merged[f"{y_col}_dcnp"].values
        ax2.scatter(x, y, label=h if h else "All", color=palette[h], alpha=0.8)
        if len(x) >= 3:
            coeffs = np.polyfit(x, y, deg=2)
            xx = np.linspace(min(x), max(x), 200)
            ax2.plot(xx, np.polyval(coeffs, xx), linestyle='--', color=palette[h], alpha=0.7)

    ax2.text(-0.15, 1.02, f"{_AZ[len(plot_data)].lower()}",
             transform=ax2.transAxes,
             fontsize=14, fontweight='bold', 
             va='top', ha='left')

    ax2.set_title("ΔFR DCN-P vs Purkinje", fontsize=14, fontweight="bold")
    ax2.axhline(0, linestyle='--', color='gray', alpha=0.5)
    ax2.axvline(0, linestyle='--', color='gray', alpha=0.5)
    ax2.axvline(-1, linestyle='--', color='gray', alpha=0.5)
    ax2.legend()
    ax2.set_xlabel("Δ Firing Rate Purkinje (Hz)", fontsize=12)
    ax2.set_ylabel("Δ Firing Rate DCN-P (Hz)", fontsize=12)
    ax2.tick_params(labelsize=11)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    for spine in ax2.spines.values():
        spine.set_linewidth(1.2)

    if save:
        fname2 = f"./tempPlots/deltaFR_purk_vs_dcnp_autor.png"
        plt.savefig(_prefix_output(fname2), dpi=200, bbox_inches="tight")
        print(f"📈 Salvato: {fname2}")

    plt.show()
    return df_exc, df_inh


In [ ]:
def plotFR_updated_v3(df_summary, hue=None, y_col='firing_rate_mean', yerr_col='firing_rate_sem', save=True):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os
    from string import ascii_uppercase as _AZ
    from scipy.stats import spearmanr

    pretty_names = {
        "granule": "Granule Cells",
        "golgi": "Golgi Cells",
        "purkinje": "Purkinje Cells",
        "dcn_p": "DCN-P",
    }

    def compute_summary(df, subset, hue=None):
        filtered = df[df["pop"].isin(subset)]
        if hue:
            grp = filtered.groupby(["SZ Factor (%)", hue]).agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        else:
            grp = filtered.groupby("SZ Factor (%)").agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        return grp


    def plot_with_corr(ax, df, x_col, y_col, yerr_col=None, label="", hue=None):
        from scipy.stats import spearmanr
        import numpy as np
    
        stats = []
        if hue:
            hue_levels = df[hue].unique()
            palette = {
                "With Resilience": "red",
                "Without Resilience": "blue",
                None: "black"
            }
            for i, h in enumerate(hue_levels):
                d = df[df[hue] == h].sort_values(by=x_col)
                x, y = d[x_col].values, d[y_col].values
                yerr = np.abs(d[yerr_col].values) if yerr_col and yerr_col in d.columns else None
                color = palette.get(h, f"C{i}")
                ax.errorbar(x, y, yerr=yerr, fmt='o-', label=f"{h}", color=color, capsize=4)
                if len(x) >= 3:
                    coeffs = np.polyfit(x, y, deg=5)
                    x_fit = np.linspace(min(x), max(x), 200)
                    ax.plot(x_fit, np.polyval(coeffs, x_fit), linestyle='-', linewidth=10, color=color, alpha=0.2)
                if 0 in x:
                    ax.axhline(y=y[x == 0][0], color="black", linestyle="--", linewidth=1.5, alpha=0.5)
                if len(x) >= 2:
                    rho, p = spearmanr(x, y)
                    stats.append((h, rho, p, color))
        else:
            d = df.sort_values(by=x_col)
            x, y = d[x_col].values, d[y_col].values
            yerr = d[yerr_col].values if yerr_col else None
            color = "black"
            ax.errorbar(x, y, yerr=yerr, fmt='o-', color=color, label=label, capsize=4)
            if len(x) >= 3:
                coeffs = np.polyfit(x, y, deg=5)
                x_fit = np.linspace(min(x), max(x), 200)
                ax.plot(x_fit, np.polyval(coeffs, x_fit), linestyle='--', color=color, alpha=1)
            if 0 in x:
                ax.axhline(y=y[x == 0][0], color="black", linestyle="--", linewidth=1.5, alpha=0.5)
            if len(x) >= 2:
                rho, p = spearmanr(x, y)
                stats.append((None, rho, p, color))
    
        # ----- OFFSET verticale di ~1 cm -----
        offset = 0.04*4   # ~1 cm nelle figure standard
    
        # ----- TITOLO (alzato) -----
        ax.set_title(label, fontsize=14, fontweight='bold', pad=30)  # pad aumenta la distanza dall’axes
        # pad=20 ≈ 0.035–0.04 nell’axes space, quindi esattamente ~1 cm
    
        # ----- STATISTICHE centrate, spostate in su -----
        if stats:
            y0 = 0.96 + offset   # alzato di 1 cm
            dy = 0.08
    
            for i, (_, rho, p, col) in enumerate(stats):
                txt = f"ρ={rho:.2f}, p={p:.2g}"
                ax.text(
                    0.5, y0 - i*dy, txt,
                    transform=ax.transAxes,
                    fontsize=11,
                    color=col,
                    va='top',
                    ha='center'
                )
      
        ax.set_xlabel("SZ Factor (%)")
        ax.set_ylabel(r"$\Delta f$ (Hz)")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    
      
    df_summary = df_summary[df_summary["pop"] != "dcn_i"]

    df_all = compute_summary(df_summary, df_summary["pop"].unique(), hue)
    df_exc = compute_summary(df_summary, ["granule"], hue)
    df_inh = compute_summary(df_summary, ["golgi"], hue)

    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(11, 13))
    fig.subplots_adjust(hspace=0.5, wspace=0.3)
    axes = axes.flatten()

    df_ml_inh = df_summary[df_summary["pop"].isin(["basket", "stellate"])]
    if hue:
        df_ml_inh_grp = df_ml_inh.groupby(["SZ Factor (%)", hue]).agg(
            y_mean=(y_col, "mean"),
            y_sem=(yerr_col, "mean") if yerr_col in df_ml_inh.columns else (y_col, "sem")
        ).reset_index()
    else:
        df_ml_inh_grp = df_ml_inh.groupby("SZ Factor (%)").agg(
            y_mean=(y_col, "mean"),
            y_sem=(yerr_col, "mean") if yerr_col in df_ml_inh.columns else (y_col, "sem")
        ).reset_index()

    plot_data = [
        (df_all, "All cells"),
        (compute_summary(df_summary, ["granule"], hue), pretty_names["granule"]),
        (compute_summary(df_summary, ["golgi"], hue), pretty_names["golgi"]),
        (df_ml_inh_grp, "MLIs"),
        (compute_summary(df_summary, ["purkinje"], hue), pretty_names["purkinje"]),
        (compute_summary(df_summary, ["dcn_p"], hue), pretty_names["dcn_p"]),
    ]

    for i, (ax, (df_plot, label)) in enumerate(zip(axes, plot_data)):
        plot_with_corr(ax, df_plot, "SZ Factor (%)", "y_mean", "y_sem", label, hue)
        ax.text(-0.15, 1.02, f"{_AZ[i].lower()}",
                transform=ax.transAxes,
                fontsize=14, fontweight='bold',
                va='top', ha='left')

    if hue:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.97), fontsize=12)

    if save:
        os.makedirs("./tempPlots", exist_ok=True)
        fname = f"./tempPlots/firing_rate_{y_col}_autor.png"
        plt.savefig(_prefix_output(fname), dpi=200, bbox_inches="tight")
        print(f"📈 Salvato: {fname}")

    purk = df_summary[df_summary["pop"] == "purkinje"]
    dcnp = df_summary[df_summary["pop"] == "dcn_p"]

    fig2, ax2 = plt.subplots(figsize=(6, 5))
    hue_levels = df_summary[hue].unique() if hue else [None]
    palette = {
        "With Resilience": "red",
        "Without Resilience": "blue",
        None: "black"
    }

    stats_pc_dcn = []
    for h in hue_levels:
        purk_sub = purk[purk[hue] == h] if h else purk
        dcnp_sub = dcnp[dcnp[hue] == h] if h else dcnp
        merged = pd.merge(
            purk_sub[["SZ Factor (%)", y_col]],
            dcnp_sub[["SZ Factor (%)", y_col]],
            on="SZ Factor (%)",
            suffixes=("_purk", "_dcnp")
        )
        x, y = merged[f"{y_col}_purk"].values, merged[f"{y_col}_dcnp"].values
        ax2.scatter(x, y, label=h if h else "All", color=palette[h], alpha=0.8)
        if len(x) >= 3:
            coeffs = np.polyfit(x, y, deg=2)
            xx = np.linspace(min(x), max(x), 200)
            ax2.plot(xx, np.polyval(coeffs, xx), linestyle='--', color=palette[h], alpha=0.7)
        if len(x) >= 2:
            rho, p = spearmanr(x, y)
            stats_pc_dcn.append((str(h) if h else "All", rho, p))

    if stats_pc_dcn:
        stat_str = "; ".join([f"{name}: ρ={rho:.2f}, p={p:.2g}" for name, rho, p in stats_pc_dcn])
        title_pc_dcn = f"ΔFR DCN-P vs Purkinje – {stat_str}"
    else:
        title_pc_dcn = "ΔFR DCN-P vs Purkinje"

    ax2.text(-0.15, 1.02, f"{_AZ[len(plot_data)].lower()}",
             transform=ax2.transAxes,
             fontsize=14, fontweight='bold',
             va='top', ha='left')

    ax2.set_title(title_pc_dcn, fontsize=14, fontweight="bold")
    ax2.axhline(0, linestyle='--', color='gray', alpha=0.5)
    ax2.axvline(0, linestyle='--', color='gray', alpha=0.5)
    ax2.axvline(-1, linestyle='--', color='gray', alpha=0.5)
    ax2.legend()
    ax2.set_xlabel("Δ Firing Rate Purkinje (Hz)", fontsize=12)
    ax2.set_ylabel("Δ Firing Rate DCN-P (Hz)", fontsize=12)
    ax2.tick_params(labelsize=11)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    for spine in ax2.spines.values():
        spine.set_linewidth(1.2)

    if save:
        fname2 = f"./tempPlots/deltaFR_purk_vs_dcnp_autor.png"
        plt.savefig(_prefix_output(fname2), dpi=200, bbox_inches="tight")
        print(f"📈 Salvato: {fname2}")

    plt.show()
    return df_exc, df_inh


In [ ]:
def plotFR_updated_v4(df_summary, hue=None, y_col='firing_rate_mean', yerr_col='firing_rate_sem', save=True):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os
    from string import ascii_uppercase as _AZ
    from scipy.stats import spearmanr

    pretty_names = {
        "granule": "Granule Cells",
        "golgi": "Golgi Cells",
        "purkinje": "Purkinje Cells",
        "dcn_p": "DCN-P",
        "basket": "Basket Cells",
        "stellate": "Stellate Cells",
    }

    def compute_summary(df, subset, hue=None):
        filtered = df[df["pop"].isin(subset)]
        if hue:
            grp = filtered.groupby(["SZ Factor (%)", hue]).agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        else:
            grp = filtered.groupby("SZ Factor (%)").agg(
                y_mean=(y_col, "mean"),
                y_sem=(yerr_col, "mean") if yerr_col in df.columns else (y_col, "sem")
            ).reset_index()
        return grp

    def plot_with_corr(ax, df, x_col, y_col, yerr_col=None, label="", hue=None):
        from scipy.stats import spearmanr
        import numpy as np
    
        stats = []
        if hue:
            hue_levels = df[hue].unique()
            palette = {
                "With Resilience": "red",
                "Without Resilience": "blue",
                None: "black"
            }
            for i, h in enumerate(hue_levels):
                d = df[df[hue] == h].sort_values(by=x_col)
                x, y = d[x_col].values, d[y_col].values
                yerr = np.abs(d[yerr_col].values) if yerr_col and yerr_col in d.columns else None
                color = palette.get(h, f"C{i}")
                ax.errorbar(x, y, yerr=yerr, fmt='o-', label=f"{h}", color=color, capsize=4)
                if len(x) >= 3:
                    coeffs = np.polyfit(x, y, deg=5)
                    x_fit = np.linspace(min(x), max(x), 200)
                    ax.plot(x_fit, np.polyval(coeffs, x_fit), linestyle='-', linewidth=10, color=color, alpha=0.2)
                if 0 in x:
                    ax.axhline(y=y[x == 0][0], color="black", linestyle="--", linewidth=1.5, alpha=0.5)
                if len(x) >= 2:
                    rho, p = spearmanr(x, y)
                    stats.append((h, rho, p, color))
        else:
            d = df.sort_values(by=x_col)
            x, y = d[x_col].values, d[y_col].values
            yerr = d[yerr_col].values if yerr_col else None
            color = "black"
            ax.errorbar(x, y, yerr=yerr, fmt='o-', color=color, label=label, capsize=4)
            if len(x) >= 3:
                coeffs = np.polyfit(x, y, deg=5)
                x_fit = np.linspace(min(x), max(x), 200)
                ax.plot(x_fit, np.polyval(coeffs, x_fit), linestyle='--', color=color, alpha=1)
            if 0 in x:
                ax.axhline(y=y[x == 0][0], color="black", linestyle="--", linewidth=1.5, alpha=0.5)
            if len(x) >= 2:
                rho, p = spearmanr(x, y)
                stats.append((None, rho, p, color))
    
        offset = 0.04*4  # ~1 cm
    
        ax.set_title(label, fontsize=14, fontweight='bold', pad=30)
    
        if stats:
            y0 = 0.96 + offset
            dy = 0.08
            for i, (_, rho, p, col) in enumerate(stats):
                txt = f"ρ={rho:.2f}, p={p:.2g}"
                ax.text(
                    0.5, y0 - i*dy, txt,
                    transform=ax.transAxes,
                    fontsize=11,
                    color=col,
                    va='top',
                    ha='center'
                )
      
        ax.set_xlabel("SZ Factor (%)")
        ax.set_ylabel(r"$\Delta f$ (Hz)")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    
    # --- escludi dcn_i ---
    df_summary = df_summary[df_summary["pop"] != "dcn_i"]

    # questi li tieni per l'output
    df_exc = compute_summary(df_summary, ["granule"], hue)
    df_inh = compute_summary(df_summary, ["golgi"], hue)

    # --- figure a 3x2 pannelli ---
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(11, 13))
    fig.subplots_adjust(hspace=0.75, wspace=0.3)
    axes = axes.flatten()

    plot_data = [
        (compute_summary(df_summary, ["granule"],  hue), pretty_names["granule"]),
        (compute_summary(df_summary, ["golgi"],    hue), pretty_names["golgi"]),
        (compute_summary(df_summary, ["basket"],   hue), pretty_names["basket"]),
        (compute_summary(df_summary, ["stellate"], hue), pretty_names["stellate"]),
        (compute_summary(df_summary, ["purkinje"], hue), pretty_names["purkinje"]),
        (compute_summary(df_summary, ["dcn_p"],    hue), pretty_names["dcn_p"]),
    ]


    # --- loop pannelli + lettere a–f in bold ---
    for i, (ax, (df_plot, label)) in enumerate(zip(axes, plot_data)):
        plot_with_corr(ax, df_plot, "SZ Factor (%)", "y_mean", "y_sem", label, hue)
        ax.text(
            -0.15, 1.02, f"{_AZ[i].lower()}",
            transform=ax.transAxes,
            fontsize=14, fontweight='bold',
            va='top', ha='left'
        )

    # legenda globale se c'è hue (With/Without Resilience, ecc.)
    if hue:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', ncol=2,
                   bbox_to_anchor=(0.5, 0.97), fontsize=12)

    if save:
        os.makedirs("./tempPlots", exist_ok=True)
        fname = f"./tempPlots/firing_rate_{y_col}_autor.png"
        plt.savefig(_prefix_output(fname), dpi=200, bbox_inches="tight")
        print(f"📈 Salvato: {fname}")

    if save:
        fname2 = f"./tempPlots/deltaFR_purk_vs_dcnp_autor.png"
        plt.savefig(_prefix_output(fname2), dpi=200, bbox_inches="tight")
        print(f"📈 Salvato: {fname2}")

    plt.show()
    return df_exc, df_inh


# path

In [ ]:
paths = [
    './20251120172014_basal_activity_2_0_glutamate_all/B_collective_results/', # NR ADULT AUTORIT
    './20251120193505_basal_activity_15_0_glutamate_all/B_collective_results/', # HR ADULT AUTORIT

]


In [ ]:
ls


# functions

## firing rates

In [ ]:
from pathlib import Path
import pandas as pd

dfs = []

for path_str in paths:
    path = Path(path_str)
    if path.exists():
        df = pd.read_pickle(f'{path}/2.functions/df_summary.pkl',)
        # Estrai il nome della directory principale
        for part in path.parts:
            if part.startswith("2025") and "basal_activity" in part:
                tokens = part.split("_")
                if len(tokens) >= 4:
                    df["Resilience"] = tokens[3]
                    df["Compensation"] = tokens[4]
                    if len(tokens) >= 6:
                        df["szh"] = tokens[5]
                        df["CellType"]=tokens[6]
                        df["dfType"] = f'{tokens[3]}_{tokens[4]}_{tokens[5]}_{tokens[6]}'
                    else:
                        df["szh"] = None
                        df["dfType"] = f'{tokens[3]}_{tokens[4]}'
        # Calcolo delta f e percentuale
        df = compute_delta_fr_with_percent_and_zscore(df)
        dfs.append(df)
    else:
        print(f"❌ File non trovato: {path}")

# Unione
if dfs:
    df_all_fr = pd.concat(dfs, ignore_index=True)
    print(f"✅ Unito {len(dfs)} file. Forma totale: {df_all_fr.shape}")
else:
    print("❌ Nessun file valido caricato.")


In [ ]:
def map_df_type(label):
    if label.startswith('15'):
        return 'With Resilience'
    elif label.startswith('2'):
        return 'Without Resilience'
    else:
        return 'Unknown'

df_all_fr['dfType'] = df_all_fr['dfType'].apply(map_df_type)


In [ ]:
df_all_fr['SZ Factor (%)'] = df_all_fr['Atrophy Level (%)'] 

In [ ]:
df_all_fr.keys()

In [ ]:
df_exc, df_inh = plotFR_updated_v4(df_all_fr, y_col='delta_fr_mean', yerr_col='delta_fr_sem', hue='dfType')


In [ ]:
df_inh

# structures

In [ ]:
import os
import yaml
import pandas as pd
import re
from glob import glob

def find_yaml_files(base_path):
    yaml_dict = {}
    pattern = os.path.join(base_path, "*_deg_*", "2.stimulus", "*.yaml")
    for file_path in glob(pattern, recursive=True):
        fname = os.path.basename(file_path)
        # estrae nome base (tutto tranne _<k>.yaml)
        m = re.match(r"(.+)_([\d.]+)\.yaml$", fname)
        if not m:
            continue
        base_name, k_str = m.groups()
        k_val = float(k_str)
        yaml_dict.setdefault(base_name, []).append((k_val, file_path))
    return yaml_dict

def flatten_dict(d, parent_key=''):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}/{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key))
        else:
            items[new_key] = v
    return items

def load_and_stack(base_name, file_list):
    rows = []
    for k_val, path in sorted(file_list, key=lambda x: x[0]):
        with open(path, 'r') as f:
            data = yaml.safe_load(f)
        flat = flatten_dict(data)
        flat['k'] = k_val
        rows.append(flat)
    df = pd.DataFrame(rows)
    df['name'] = base_name
    return df

# === ESEMPIO D'USO ===
base_path = "/home/alberto/cerebellum/cerebellum/20250625142436_basal_activity_1_0_glutamate_all"
groups = find_yaml_files(base_path)

# `dfs` è un dict: nome_base → DataFrame
dfs = {name: load_and_stack(name, flist) for name, flist in groups.items()}




In [ ]:
from pathlib import Path
import pandas as pd
import yaml
import seaborn as sns
import matplotlib.pyplot as plt
import os


dfs = []

for path_str in paths:
    path = Path(path_str)
    if path.exists():
        df = pd.read_pickle(f'{path}/1.structures/df_summary.pkl',)
        for part in path.parts:
            if part.startswith("2025") and "basal_activity" in part:
                tokens = part.split("_")
                if len(tokens) >= 4:
                    df["Resilience"] = tokens[3]
                    df["Compensation"] = tokens[4]
                    if len(tokens) >= 6:
                        df["szh"] = tokens[5]
                        df["CellType"] = tokens[6]
                        df["dfType"] = f'{tokens[3]}_{tokens[4]}_{tokens[5]}_{tokens[6]}'
                    else:
                        df["szh"] = None
                        df["dfType"] = f'{tokens[3]}_{tokens[4]}'
        dfs.append(df)

df_all_str = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# === STEP 2: Parsing YAML e costruzione df_all_weights ===
from pathlib import Path
import yaml
import pandas as pd

base_dirs = [Path(p).parents[0] for p in paths]

yaml_files = []
for base_dir in base_dirs:
    for pattern in ["*/2.stimulus/basal_vitro_*.yaml",
                    "*/2.stimulus/dcn_vitro_nest_*.yaml",
                    "*/2.stimulus/dcn_io_vitro_nest_*.yaml"]:
        yaml_files.extend(list(base_dir.glob(pattern)))

def extract_k_from_dir(dir_path):
    for part in dir_path.parts:
        if "deg" in part:
            try:
                return float(part.split('_')[-1])
            except ValueError:
                return None
    return None

all_rows = []
for fpath in yaml_files:
    k_val = extract_k_from_dir(fpath.parent.parent)
    try:
        with open(fpath, "r") as f:
            stim = yaml.safe_load(f)
    except Exception as e:
        print(f"❌ Errore nel parsing del file {fpath}: {e}")
        continue

    sim_dict = stim.get('simulations', {})
    if 'basal_activity' not in sim_dict:
        print(f"⚠️ File {fpath.name} non contiene 'basal_activity'. Skippato.")
        continue

    stim_key = list(sim_dict.keys())[0]
    conn_models = sim_dict.get(stim_key, {}).get('connection_models', {})
    for conn, data in conn_models.items():
        weight = data.get('synapse', {}).get('weight', None)
        all_rows.append({
            'k': k_val,
            'connection': conn,
            'weight': weight,
            'stim_name': stim_key,
            'source_file': fpath.name
        })

df_all_weights = pd.DataFrame(all_rows)

# === STEP 3: Mappatura presinaptica e merge con firing rate ===
connection_to_presyn = {
    'ascending_axon_to_golgi': 'granule',
    'ascending_axon_to_purkinje': 'granule',
    'basket_to_basket': 'basket',
    'basket_to_purkinje': 'basket',
    'gap_goc': 'golgi',
    'glomerulus_to_golgi': 'glomerulus',
    'glomerulus_to_granule': 'glomerulus',
    'golgi_to_glomerulus': 'golgi',
    'golgi_to_golgi': 'golgi',
    'mossy_fibers_to_glomerulus': 'mossy',
    'parallel_fiber_to_basket': 'granule',
    'parallel_fiber_to_golgi': 'granule',
    'parallel_fiber_to_purkinje': 'granule',
    'parallel_fiber_to_stellate': 'granule',
    'stellate_to_purkinje': 'stellate',
    'stellate_to_stellate': 'stellate',
    'io_to_purkinje': 'io'
}

df_all_weights['presyn_pop'] = df_all_weights['connection'].map(connection_to_presyn)

# Mappatura tipo di sinapsi: eccitatoria (glutamate) o inibitoria (GABA)
connection_to_type = {
    'ascending_axon_to_golgi': 'excitatory (glutamate)',
    'ascending_axon_to_purkinje': 'excitatory (glutamate)',
    'basket_to_basket': 'inhibitory (GABA)',
    'basket_to_purkinje': 'inhibitory (GABA)',
    'gap_goc': 'electrical',
    'glomerulus_to_golgi': 'excitatory (glutamate)',
    'glomerulus_to_granule': 'excitatory (glutamate)',
    'golgi_to_glomerulus': 'inhibitory (GABA)',
    'golgi_to_golgi': 'inhibitory (GABA)',
    'mossy_fibers_to_glomerulus': 'excitatory (glutamate)',
    'parallel_fiber_to_basket': 'excitatory (glutamate)',
    'parallel_fiber_to_golgi': 'excitatory (glutamate)',
    'parallel_fiber_to_purkinje': 'excitatory (glutamate)',
    'parallel_fiber_to_stellate': 'excitatory (glutamate)',
    'stellate_to_purkinje': 'inhibitory (GABA)',
    'stellate_to_stellate': 'inhibitory (GABA)',
    'io_to_purkinje': 'inhibitory (GABA)'
}

df_all_weights['synType'] = df_all_weights['connection'].map(connection_to_type)





In [ ]:
df_merged = df_all_weights.merge(
    df_all_fr[['pop', 'firing_rate_mean']],
    left_on='presyn_pop',
    right_on='pop',
    how='left'
).drop(columns='pop')

# Merge finale con df_all_str
if 'connection' in df_all_str.columns and 'k' in df_all_str.columns:
    df_all_str = df_all_str.merge(df_merged, on=['connection', 'k'], how='left')

# === STEP 4 (opzionale): Calcolo effective input ===
if "effective_input" not in df_all_str.columns:
    df_all_str["effective_synapses"] = df_all_str["synapses"] * df_all_str["weight"]
    
df_all_str["effective_synapses"] = df_all_str["synapses"] * df_all_str["weight"]
df_all_str["effective_convergence"] = df_all_str["convergence"] * df_all_str["weight"]
df_all_str["effective_divergence"] = df_all_str["divergence"] * df_all_str["weight"]
df_all_str["effective_syn_per_pair"] = df_all_str["syn_per_pair"] * df_all_str["weight"]
df_all_str["SZ Factor (%)"] = df_all_str["Atrophy Factor (%)"] 

def map_df_type(label):
    if label.startswith('15'):
        return 'With Resilience'
    elif label.startswith('2'):
        return 'Without Resilience'
    else:
        return 'Unknown'

df_all_str['dfType'] = df_all_str['dfType'].apply(map_df_type)
df_all_str.keys()


In [ ]:
df_all_str.keys()

In [ ]:
plot_metrics_merge_from_df_summary(df_all_str, 'SZ Factor (%)', 'SZ Factor (%)', hue="dfType", outname="merged_structures")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import numpy as np
import os

# === Font settings ===
font_size = 16
plt.rcParams.update({'font.size': font_size})

# === Colori ===
custom_palette = {
    'Without Resilience': 'blue',
    'With Resilience': 'red'
}

# === Prepara i subplot: 2 righe × 4 colonne ===
fig, axes = plt.subplots(2, 4, figsize=(25, 13))
axes = axes.flatten()

# === 1. Synaptic Weight ===
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='weight',
    hue='synType',
    estimator=np.mean,
    palette=['orange', 'k'],
    ax=axes[0]
)
axes[0].set_yscale('log')
axes[0].set_xlabel("Synaptic Scaling (%)")
axes[0].set_ylabel("Synaptic Weight (nS)")
axes[0].set_title("Synaptic Weight")
axes[0].legend(title='Synapse Type', loc='best')

# === 2. DCI ===
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='DCI',
    hue='cell_name',
    ax=axes[1]
)
axes[1].set_yscale('log')
axes[1].set_xlabel("Cell Scaling (%)\n[shrinking/pruning/thinning]")
axes[1].set_ylabel("DCI (a.u.)")
axes[1].set_title("DCI per Cell Type")
axes[1].legend(title='Cell Type', loc='best')

# === 3. Resilience Function ===
k_values = np.linspace(1.0, 0, 300)
sz_factor = (1 - k_values) * 100
sharpness_values = np.arange(0, 100)
norm = colors.Normalize(vmin=sharpness_values.min(), vmax=sharpness_values.max())
cmap = cm.Greys_r

def k_Resilience_simple_v3(k, k0=0.75, sharpness=20):
    k = np.array(k, dtype=float)
    raw = 1 / (1 + np.exp(-sharpness * (k - k0)))
    raw0 = 1 / (1 + np.exp(-sharpness * (0 - k0)))
    raw1 = 1 / (1 + np.exp(-sharpness * (1 - k0)))
    return (raw - raw0) / (raw1 - raw0 + 1e-9)

ax = axes[2]
for sharp in sharpness_values:
    y = k_Resilience_simple_v3(k_values, k0=0.5, sharpness=sharp)*100
    if sharp == 2:
        ax.plot(sz_factor, y, color='blue', label='Without Resilience', linewidth=1)
    elif sharp == 15:
        ax.plot(sz_factor, y, color='red', label='With Resilience', linewidth=1)
    else:
        ax.plot(sz_factor, y, color=cmap(norm(sharp)), alpha=0.1, linewidth=1)

#y = k_Resilience_simple_v3(k_values, k0=1, sharpness=3)*100
#ax.plot(sz_factor, y, color='red', label='low Resilience', linewidth=1)

ax.axhline(y=50, linestyle='--', color='gray', linewidth=1)
ax.axvline(x=50, linestyle='--', color='gray', linewidth=1)
ax.axvline(x=40, linestyle='--', color='orange', linewidth=1)
ax.axhline(y=100, linestyle='--', color='orange', linewidth=1)
#ax.text(sz_factor.min()-10, 120, "Initial Brain Reserve", fontsize=font_size, color='black', va='bottom')

ax.set_xlabel("Cell Scaling (%)\n[Shrinking/Pruning/Thinning]")
ax.set_ylabel("Cell Resilience to CS (%)\n[thickness/density]")
ax.set_title("Resilience Function")
ax.legend(loc='lower left')
ax.grid(axis='y')

# === 4. Total Cells ===
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='total_cells',
    hue='dfType',
    palette=custom_palette,
    ax=axes[3]
)
axes[3].set_xlabel("Cell Scaling (%)\n[Shrinking/Pruning/Thinning]")
axes[3].set_ylabel("Total Cells")
axes[3].set_title("Total Cells")
axes[3].legend(title='Brain Resilience', loc='best')

# === 5. Synapse Count ===
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='synapses',
    hue='dfType',
    palette=custom_palette,
    ax=axes[4]
)
axes[4].set_xlabel("Cell Scaling (%)\n[Shrinking/Pruning/Thinning]")
axes[4].set_ylabel("Mean Synapses (#)")
axes[4].set_title("Synapse Count")
axes[4].legend(title='Brain Resilience', loc='best')

# === 6. Convergence ===
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='convergence',
    hue='dfType',
    palette=custom_palette,
    ax=axes[5]
)
axes[5].set_xlabel("Cell Scaling (%)\n[Shrinking/Pruning/Thinning]")
axes[5].set_ylabel("Mean Convergence (#)")
axes[5].set_title("Convergence")
axes[5].legend(title='Brain Resilience', loc='best')

# === 7. Divergence ===
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='divergence',
    hue='dfType',
    estimator=np.median,
    palette=custom_palette,
    ax=axes[6]
)
axes[6].set_xlabel("Cell Scaling (%)\n[Shrinking/Pruning/Thinning]")
axes[6].set_ylabel("Mean Divergence (#)")
axes[6].set_title("Divergence")
axes[6].legend(title='Brain Resilience', loc='best')

# === 8. Synapses per Pair ===
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='syn_per_pair',
    hue='dfType',
    palette=custom_palette,
    ax=axes[7]
)
axes[7].set_xlabel("Cell Scaling (%)\n[Shrinking/Pruning/Thinning]")
axes[7].set_ylabel("Synapses per Pair (#)")
axes[7].set_title("Synapses per Pair")
axes[7].legend(title='Brain Resilience', loc='best')

# === Layout finale ===
plt.tight_layout(h_pad=3.5)  # Aumenta lo spazio verticale tra righe
os.makedirs("./tempPlots", exist_ok=True)
plt.savefig(_prefix_output("./tempPlots/multi_panel_summary_extended_autor.png"), dpi=300)
plt.show()



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import numpy as np
import os

plt.rcParams.update({'font.size': 16})

custom_palette = {'Without Resilience': 'blue', 'With Resilience': 'red'}

fig, axes = plt.subplots(4, 2, figsize=(12, 21))
axes = axes.flatten()

# === Add panel letters ===
panel_letters = ['a','b','c','d','e','f','g','h']
for i, ax in enumerate(axes):
    ax.text(-0.12, 1.10, panel_letters[i], transform=ax.transAxes,
            fontsize=20, fontweight='bold', va='top', ha='right')

# --- Synaptic Weight ---
sns.lineplot(data=df_all_str, x='SZ Factor (%)', y='weight', hue='synType',
             estimator=np.mean, palette=['orange','k'], ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_xlabel("SZ Factor (%)")
axes[0].set_ylabel("Synaptic Weight (nS)")
axes[0].set_title("Synaptic Weight")
axes[0].legend(title=None)

# --- DCI ---
sns.lineplot(data=df_all_str, x='SZ Factor (%)', y='DCI', hue='cell_name', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_xlabel("SZ Factor (%)")
axes[1].set_ylabel("DCI (a.u.)")
axes[1].set_title("DCI per Cell Type")
axes[1].legend(title=None)

# --- Resilience Function ---
k_values = np.linspace(1.0, 0, 300)
sz_factor = (1 - k_values) * 100
sharpness_values = np.arange(0, 100)
norm = colors.Normalize(vmin=sharpness_values.min(), vmax=sharpness_values.max())
cmap = cm.Greys_r

def k_Resilience_simple_v3(k, k0=0.5, sharpness=20):
    k = np.array(k, dtype=float)
    raw = 1 / (1 + np.exp(-sharpness * (k - k0)))
    raw0 = 1 / (1 + np.exp(-sharpness * (0 - k0)))
    raw1 = 1 / (1 + np.exp(-sharpness * (1 - k0)))
    return (raw - raw0) / (raw1 - raw0 + 1e-9)

ax = axes[2]
for sharp in sharpness_values:
    y = k_Resilience_simple_v3(k_values, k0=0.5, sharpness=sharp)*100
    if sharp == 2:
        ax.plot(sz_factor, y, color='blue', label='Without Resilience')
    elif sharp == 15:
        ax.plot(sz_factor, y, color='red', label='With Resilience')
    else:
        ax.plot(sz_factor, y, color=cmap(norm(sharp)), alpha=0.1)

ax.axhline(50, ls='--', color='gray')
ax.axvline(50, ls='--', color='gray')
ax.axvline(40, ls='--', color='orange')
ax.axhline(100, ls='--', color='orange')

ax.set_xlabel("SZ Factor (%)")
ax.set_ylabel("Cell Resilience to CS (%)")
ax.set_title("Resilience Function")
ax.legend(loc='lower left', title=None)

# --- Total Cells ---
sns.lineplot(data=df_all_str, x='SZ Factor (%)', y='total_cells',
             hue='dfType', palette=custom_palette, ax=axes[3])
axes[3].set_xlabel("SZ Factor (%)")
axes[3].set_ylabel("Total Cells")
axes[3].set_title("Total Cells")
axes[3].legend(title=None)

# --- Synapse Count ---
sns.lineplot(data=df_all_str, x='SZ Factor (%)', y='synapses',
             hue='dfType', palette=custom_palette, ax=axes[4])
axes[4].set_xlabel("SZ Factor (%)")
axes[4].set_ylabel("Mean Synapses (#)")
axes[4].set_title("Synapse Count")
axes[4].legend(title=None)

# --- Convergence ---
sns.lineplot(data=df_all_str, x='SZ Factor (%)', y='convergence',
             hue='dfType', palette=custom_palette, ax=axes[5])
axes[5].set_xlabel("SZ Factor (%)")
axes[5].set_ylabel("Mean Convergence (#)")
axes[5].set_title("Convergence")
axes[5].legend(title=None)

# --- Divergence ---
sns.lineplot(data=df_all_str, x='SZ Factor (%)', y='divergence',
             hue='dfType', estimator=np.median, palette=custom_palette, ax=axes[6])
axes[6].set_xlabel("SZ Factor (%)")
axes[6].set_ylabel("Mean Divergence (#)")
axes[6].set_title("Divergence")
axes[6].legend(title=None)

# --- Synapses per Pair ---
sns.lineplot(data=df_all_str, x='SZ Factor (%)', y='syn_per_pair',
             hue='dfType', palette=custom_palette, ax=axes[7])
axes[7].set_xlabel("SZ Factor (%)")
axes[7].set_ylabel("Synapses per Pair (#)")
axes[7].set_title("Synapses per Pair")
axes[7].legend(title=None)

plt.tight_layout(h_pad=3.5)
os.makedirs("./tempPlots", exist_ok=True)
plt.savefig(_prefix_output("./tempPlots/multi_panel_summary_extended_column_autor.png"), dpi=300)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os
from matplotlib.ticker import NullFormatter, NullLocator
import pandas as pd

# Crea cartella per il salvataggio
os.makedirs("./tempPlots", exist_ok=True)

# Copia sicura del DataFrame
df_rel = df_all_str.copy()

# Calcola i baseline per ciascun tipo cellulare
baseline = df_rel[df_rel['SZ Factor (%)'] == 0].groupby('cell_name')['DCI'].mean()

# Normalizza: variazione percentuale rispetto al baseline
df_rel['DCI_rel'] = df_rel.apply(
    lambda row: (row['DCI'] / baseline.loc[row['cell_name']] - 1) * 100 if row['cell_name'] in baseline else None,
    axis=1
)

# Crea figura singola
fig, ax = plt.subplots(figsize=(9, 7))

# Plot del DCI relativo
sns.lineplot(
    data=df_rel,
    x='SZ Factor (%)',
    y='DCI_rel',
    hue='cell_name',
    ax=ax
)

# Rimuove ticks e numeri
ax.set_xticks([])
ax.set_yticks([])
ax.xaxis.set_major_formatter(NullFormatter())
ax.yaxis.set_major_formatter(NullFormatter())
ax.xaxis.set_major_locator(NullLocator())
ax.yaxis.set_major_locator(NullLocator())

# Rimuove spine dx e sopra
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Etichette assi
ax.set_xlabel("SZ Factor (%)", fontsize=18, fontweight='bold')
ax.set_ylabel("Cell Complexity (%)", fontsize=18, fontweight='bold')

# Legenda fuori dal plot
legend = ax.legend(
    title='',
    loc='upper right',
    fontsize=16
)

plt.subplots_adjust(right=0.75)

# Salva
plt.savefig(_prefix_output("./tempPlots/dci_per_celltype_relative_autor.png"), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
df_all_str.keys()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os
from matplotlib.ticker import NullFormatter, NullLocator

# Crea cartella per salvataggio
os.makedirs("./tempPlots", exist_ok=True)

# Crea figura
fig, ax = plt.subplots(figsize=(6, 5))

# Lineplot con shading ±1 std
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='convergence',
    errorbar='se',  # Usa deviazione standard per lo shading
    ax=ax
)

# Rimuove spine destra e superiore
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_yscale('log')

# Rimuove tutti i tick numerici
ax.set_xticks([])
ax.set_yticks([])
ax.yaxis.set_major_formatter(NullFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_major_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# Etichette assi
ax.set_xlabel("SZ Factor (%)", fontsize=16, fontweight='bold')
ax.set_ylabel("Convergence (#)", fontsize=16, fontweight='bold')

# Salvataggio
plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/synapse_count_autor.png"), dpi=300)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
from matplotlib.ticker import NullFormatter, NullLocator

# Crea cartella per salvataggio
os.makedirs("./tempPlots", exist_ok=True)

# Crea una copia del dataframe per evitare modifiche in-place
df_rel = df_all_str.copy()

# Calcola i baseline separatamente: median per divergence, mean per le altre
baseline = {}
for var in ['convergence', 'synapses']:
    baseline[var] = df_rel[df_rel['SZ Factor (%)'] == 0].groupby('dfType')[var].mean()
baseline['divergence'] = df_rel[df_rel['SZ Factor (%)'] == 0].groupby('dfType')['divergence'].median()

# Funzione per normalizzare ogni metrica rispetto al baseline corretto
for var in ['convergence', 'divergence', 'synapses']:
    df_rel[var + '_rel'] = df_rel.apply(
        lambda row: (row[var] / baseline[var].loc[row['dfType']] - 1) * 100,
        axis=1
    )

# Crea figura
fig, ax = plt.subplots(figsize=(6, 6))

# Definisci mappa colore manuale
palette = {
    'convergence_rel': 'dodgerblue',
    'divergence_rel': 'tomato',
    'synapses_rel': 'black'
}

# Plot per ciascuna metrica
for var in ['convergence_rel', 'divergence_rel', 'synapses_rel']:
    if var == 'divergence_rel':
        sns.lineplot(
            data=df_rel,
            x='SZ Factor (%)',
            y=var,
            label='Divergence',
            color=palette[var],
            estimator=np.median,
            ax=ax
        )
    else:
        sns.lineplot(
            data=df_rel,
            x='SZ Factor (%)',
            y=var,
            label=var.replace('_rel', '').capitalize(),
            color=palette[var],
            ax=ax
        )

# Rimuove spine destra e superiore
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Rimuove tutti i tick numerici
ax.set_xticks([])
ax.set_yticks([])
ax.yaxis.set_major_formatter(NullFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_major_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# Etichette assi
ax.set_xlabel("SZ Factor (%)", fontsize=18, fontweight='bold')
ax.set_ylabel("Connectivity (%)", fontsize=18, fontweight='bold')

# Legenda
ax.legend(title='', fontsize=18, loc='upper right')

# Salvataggio
plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/synapse_count_relative_autor.png"), dpi=300)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib.ticker import NullFormatter, NullLocator

# Crea la cartella se non esiste
os.makedirs("./tempPlots", exist_ok=True)

# Crea la figura
fig, ax = plt.subplots(figsize=(6, 6))
sns.lineplot(
    data=df_all_str,
    x='SZ Factor (%)',
    y='weight',
    hue='synType',
    estimator=np.mean,
    palette=['orange', 'k'],
    ax=ax
)

# Scala logaritmica Y
ax.set_yscale('log')

# ✅ Rimuove TUTTO sull'asse Y: numeri, tick, etichette, lineette
ax.yaxis.set_major_formatter(NullFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_major_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# Rimuove X ticks e label
ax.set_xticks([])

ax.set_xlabel("")
ax.set_ylabel("Synaptic Weight (nS)", fontsize=18, fontweight="bold")

# Etichette concettuali
ax.text(0.5, -0.05, 'SZ Factor (%)', ha='center', va='center', fontweight="bold",
        transform=ax.transAxes, fontsize=18)
#ax.text(-0.08, 0.5, 'Synaptic Weight', ha='center', va='center',
#        rotation='vertical', transform=ax.transAxes, fontsize=12)

# Titolo e legenda
ax.legend(title='', loc='upper right', fontsize=18)

# Mostra solo spine sinistra e inferiore
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Salva e mostra
plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/syn_weight_autor.png"), dpi=300)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib.ticker import NullFormatter, NullLocator

# Crea la cartella se non esiste
os.makedirs("./tempPlots", exist_ok=True)

# Calcola baseline per ciascun tipo di sinapsi a SZ = 0
baseline_weights = df_all_str[df_all_str['SZ Factor (%)'] == 0].groupby('synType')['weight'].mean()

# Normalizza rispetto al baseline
df_weight_rel = df_all_str.copy()
def normalize_weight(row):
    syn = row['synType']
    if pd.isna(syn):
        return np.nan
    return (row['weight'] / baseline_weights.loc[syn] - 1) * 100

df_weight_rel['weight_rel'] = df_weight_rel.apply(normalize_weight, axis=1)


# Crea la figura
fig, ax = plt.subplots(figsize=(6, 6))
sns.lineplot(
    data=df_weight_rel,
    x='SZ Factor (%)',
    y='weight_rel',
    hue='synType',
    estimator=np.mean,
    palette=['orange', 'k'],
    ax=ax
)

# ✅ Rimuove TUTTO sull'asse Y: numeri, tick, etichette, lineette
ax.yaxis.set_major_formatter(NullFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_major_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# Rimuove X ticks e label
ax.set_xticks([])

ax.set_xlabel("")
ax.set_ylabel("Synaptic Weight (%)", fontsize=18, fontweight="bold")

# Etichetta X come testo concettuale
ax.text(0.5, -0.05, 'SZ Factor (%)', ha='center', va='center', fontweight="bold",
        transform=ax.transAxes, fontsize=18)

# Titolo e legenda
ax.legend(title='', loc='lower left', fontsize=18)

# Mostra solo spine sinistra e inferiore
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Salva e mostra
plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/syn_weight_relative.png"), dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import os
from matplotlib.ticker import NullFormatter, NullLocator

# Crea cartella se non esiste
os.makedirs("./tempPlots", exist_ok=True)

# === Dati e funzione ===
k_values = np.linspace(1.0, 0, 300)
sz_factor = (1 - k_values) * 100

def k_Resilience_simple_v3(k, k0=0.75, sharpness=20):
    k = np.array(k, dtype=float)
    raw = 1 / (1 + np.exp(-sharpness * (k - k0)))
    raw0 = 1 / (1 + np.exp(-sharpness * (0 - k0)))
    raw1 = 1 / (1 + np.exp(-sharpness * (1 - k0)))
    return (raw - raw0) / (raw1 - raw0 + 1e-9)

# === Monoplot ===
fig, ax = plt.subplots(figsize=(6, 6))

# High Resilience → rosso
y_high = k_Resilience_simple_v3(k_values, k0=0.5, sharpness=15) * 100
ax.plot(sz_factor, y_high, color='red', label='With Resilience', linewidth=2)

# Without resilience
y_low = k_Resilience_simple_v3(k_values, k0=0.5, sharpness=2) * 100
ax.plot(sz_factor, y_low, color='blue', label='Without Resilience', linewidth=2)

# Linee guida grigie
ax.axhline(y=50, linestyle='--', color='gray', linewidth=1, alpha=0.5)
ax.axvline(x=50, linestyle='--', color='gray', linewidth=1, alpha=0.5)

# Mostra solo spine sinistra e inferiore
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Rimuove tick e numeri
ax.set_xticks([])
ax.set_yticks([])
ax.xaxis.set_major_formatter(NullFormatter())
ax.yaxis.set_major_formatter(NullFormatter())
ax.xaxis.set_major_locator(NullLocator())
ax.yaxis.set_major_locator(NullLocator())

# Etichette concettuali in grassetto
ax.text(0.5, -0.10, 'SZ Factor (%)', ha='center', va='center',
        transform=ax.transAxes, fontsize=16, fontweight="bold")
ax.text(-0.08, 0.5, 'Cerebellar Reserve', ha='center', va='center',
        rotation='vertical', transform=ax.transAxes, fontsize=18, fontweight="bold")

# ✅ Titolo sintetico e smart
plt.suptitle("Prob. of Cell Life", fontsize=18, fontweight="bold", y=0.9)

# Legenda
ax.legend(loc='upper right', fontsize=18, title_fontsize=18)

# Salva e mostra
plt.tight_layout(rect=[0, 0, 1, 0.95])  # lascia spazio per il titolo
plt.savefig(_prefix_output("./tempPlots/reserve_model_autor.png"), dpi=300)
plt.show()


In [ ]:
df_all_str.keys()

In [ ]:
df_all_str['Resilience']

# pc vs structure vs dcn

In [ ]:
df_all_str[df_all_str['cell_name']=='PurkinjeCell']['count']
df_all_str[df_all_str['cell_name']=='PurkinjeCell']['SZ Factor (%)']


In [ ]:
# --- FR: prendo sia purkinje che dcn_p (robusto: usa 'cell_type' se c'è, altrimenti 'pop')
col_type = 'cell_type' if 'cell_type' in df_all_fr.columns else 'pop'
mask_fr = df_all_fr[col_type].isin(['purkinje', 'dcn_p'])

unique_fr = (
    df_all_fr.loc[mask_fr, ['firing_rate_mean', 'SZ Factor (%)', 'Resilience', 'pop']]
    .drop_duplicates()
    .reset_index(drop=True)
).sort_values(by=['Resilience', 'SZ Factor (%)', 'pop'])

# --- STR: purkinje_cell
unique_str = (
    df_all_str.loc[df_all_str['cell_type'].eq('purkinje_cell'), ['count', 'SZ Factor (%)', 'Resilience']]
    .drop_duplicates()
    .reset_index(drop=True)
).sort_values(by=['Resilience', 'SZ Factor (%)'])

# --- Merge robusto su SZ (arrotondato) + dfTag, mantenendo anche 'pop'
dec = 6
fr_rob  = unique_fr.copy()
str_rob = unique_str.copy()
fr_rob['SZ_round']  = fr_rob['SZ Factor (%)'].astype(float).round(dec)
str_rob['SZ_round'] = str_rob['SZ Factor (%)'].astype(float).round(dec)

merged_robust = (
    pd.merge(
        fr_rob.drop(columns=['SZ Factor (%)']),
        str_rob.drop(columns=['SZ Factor (%)']),
        on=['SZ_round', 'Resilience'],
        how='inner',
        suffixes=('_fr', '_str')
    )
    .rename(columns={'SZ_round': 'SZ Factor (%)'})
    .sort_values(['Resilience', 'SZ Factor (%)', 'pop'])
    .reset_index(drop=True)
)

merged_robust

p = merged_robust.copy()

p['Resilience'] = (
    p['Resilience']
    .astype(float).astype(int)   # converte qualsiasi cosa in 2 o 15 veri
    .replace({2: 'Without Resilience', 15: 'With Resilience'})
)


In [ ]:
p

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

# p deve avere: ['firing_rate_mean','Resilience','pop','SZ Factor (%)','count']

wide=(p.pivot_table(index=['Resilience','SZ Factor (%)','count'],
                    columns='pop',values='firing_rate_mean',aggfunc='mean')
        .reset_index())
wide=wide.dropna(subset=['purkinje','dcn_p']).copy()

markers={
    'With Resilience':'o',
    'Without Resilience':'x',
}

# === CORRELAZIONI (PEARSON) E COHEN f² PER OGNI CONDIZIONE ===
corr_count={}   # corr_count[tag] = {'r':..., 'p':..., 'f2':..., 'n':...}
corr_pcfr={}    # corr_pcfr[tag]  = {'r':..., 'p':..., 'f2':..., 'n':...}

for tag in markers.keys():
    sub=wide[wide['Resilience'].eq(tag)]
    if sub.empty:
        continue

    y=sub['dcn_p']

    # DCN vs PC count
    x_c=sub['count']
    if len(sub)>=2:
        r_c,p_c=pearsonr(x_c,y)
        f2_c=(r_c**2)/(1-r_c**2) if abs(r_c)<1 else np.inf
    else:
        r_c,p_c,f2_c=np.nan,np.nan,np.nan
    corr_count[tag]={'r':r_c,'p':p_c,'f2':f2_c,'n':len(sub)}

    # DCN vs PC FR
    x_p=sub['purkinje']
    if len(sub)>=2:
        r_p,p_p=pearsonr(x_p,y)
        f2_p=(r_p**2)/(1-r_p**2) if abs(r_p)<1 else np.inf
    else:
        r_p,p_p,f2_p=np.nan,np.nan,np.nan
    corr_pcfr[tag]={'r':r_p,'p':p_p,'f2':f2_p,'n':len(sub)}

# --- stampa opzionale in plain ---
print("\n=== CORRELAZIONI PEARSON + COHEN f² PER CONDIZIONE DI RESILIENCE ===")
print("PC count → DCN")
for tag,v in corr_count.items():
    print(f"[{tag}] r = {v['r']:.3f}, p = {v['p']:.3f}, f² = {v['f2']:.3f}, n = {v['n']}")
print("\nPC FR → DCN")
for tag,v in corr_pcfr.items():
    print(f"[{tag}] r = {v['r']:.3f}, p = {v['p']:.3f}, f² = {v['f2']:.3f}, n = {v['n']}")

# === FIGURA CON DUE PANNELLI ===
fig,axes=plt.subplots(1,2,figsize=(12,5),sharey=True)

norm=mpl.colors.Normalize(vmin=wide['SZ Factor (%)'].min(),
                          vmax=wide['SZ Factor (%)'].max())
cmap='viridis'
sc=None

for tag,mk in markers.items():
    sub=wide[wide['Resilience'].eq(tag)]
    if sub.empty:
        continue

    sc=axes[0].scatter(sub['count'],sub['dcn_p'],
                       c=sub['SZ Factor (%)'],cmap=cmap,norm=norm,
                       s=80,edgecolor='k',alpha=0.9,marker=mk)
    sc=axes[1].scatter(sub['purkinje'],sub['dcn_p'],
                       c=sub['SZ Factor (%)'],cmap=cmap,norm=norm,
                       s=80,edgecolor='k',alpha=0.9,marker=mk)

if sc is not None:
    cbar=fig.colorbar(sc,ax=axes.ravel().tolist(),pad=0.02,fraction=0.046)
    cbar.set_label('SZ Factor (%)')

ymin,ymax=wide['dcn_p'].min(),wide['dcn_p'].max()

xmin_c,xmax_c=wide['count'].min(),wide['count'].max()
axes[0].plot([xmin_c,xmax_c],[ymax,ymin],'--',color='black',linewidth=1.5)

xmin_p,xmax_p=wide['purkinje'].min(),wide['purkinje'].max()
axes[1].plot([xmin_p,xmax_p],[ymax,ymin],'--',color='black',linewidth=1.5)

axes[0].set_xlabel('PC count')
axes[0].set_ylabel('DCN-P FR [Hz]')
axes[1].set_xlabel('PC Firing Rate [Hz]')

axes[0].invert_xaxis()
axes[1].invert_xaxis()

def make_title_count(cdict):
    parts=[]
    if 'With Resilience' in cdict:
        v=cdict['With Resilience']
        parts.append(
            f"With Resilience: r={v['r']:.2f}, p={v['p']:.3f}, f²={v['f2']:.2f}"
        )
    if 'Without Resilience' in cdict:
        v=cdict['Without Resilience']
        parts.append(
            f"Without Resilience: r={v['r']:.2f}, p={v['p']:.3f}, f²={v['f2']:.2f}"
        )
    return "PC count → DCN\n" + " \n ".join(parts)

def make_title_pcfr(cdict):
    parts=[]
    if 'With Resilience' in cdict:
        v=cdict['With Resilience']
        parts.append(
            f"With Resilience: r={v['r']:.2f}, p={v['p']:.3f}, f²={v['f2']:.2f}"
        )
    if 'Without Resilience' in cdict:
        v=cdict['Without Resilience']
        parts.append(
            f"Without Resilience: r={v['r']:.2f}, p={v['p']:.3f}, f²={v['f2']:.2f}"
        )
    return "PC FR → DCN\n" + " \n ".join(parts)

axes[0].set_title(make_title_count(corr_count),fontweight='bold',fontsize=11)
axes[1].set_title(make_title_pcfr(corr_pcfr),fontweight='bold',fontsize=11)

for ax in axes:
    ax.grid(True,linestyle='--',alpha=0.4)

handles=[]
labels=[]
for tag,mk in markers.items():
    h=Line2D([0],[0],marker=mk,linestyle='None',
             markerfacecolor='none',markeredgecolor='k')
    handles.append(h)
    labels.append(tag)

fig.legend(handles,labels,title="Resilience",
           loc='center left',bbox_to_anchor=(0.325,-0.125),
           borderaxespad=0.)

plt.tight_layout(rect=[0,0,0.8,1])
axes[0].text(-0.1,1.15,'a',transform=axes[0].transAxes,
             fontweight='bold',va='top',ha='left')
axes[1].text(-0.1,1.15,'b',transform=axes[1].transAxes,
             fontweight='bold',va='top',ha='left')

fname="./tempPlots/pcnumberdcn_2panels_corr_byRes_autor.png"
plt.savefig(_prefix_output(fname),dpi=120,bbox_inches="tight")
print(f"📈 Salvato: {fname}")
